# Explore ADSL - CDISC pilot study (CDISCPILOT01)
ADSL is the ADaM subject-level analysis dataset: one row per subject, with demographics, treatment, population flags and disposition. Every table in a clinical study report starts from it, so a stats programmer checks it before building anything.

This notebook reads ADSL from the project's NetApp Volume, runs some quick checks, and saves a few plots to `artifacts/`.

In [ ]:
import os, sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append(os.environ.get("DOMINO_WORKING_DIR", os.path.abspath("../..")))
from common import paths
from common.adam import load_adsl

plots = paths.artifact_dir("a_explore_adsl")
print("Reading", paths.ADSL_FILE)

## Load ADSL
The file is a SAS `sas7bdat` dataset. pandas reads it directly - no SAS licence needed.

In [ ]:
adsl = load_adsl()
print(f"{len(adsl)} subjects, {adsl.shape[1]} variables")
adsl.head()

## What is in the dataset?

In [ ]:
summary = pd.DataFrame({"type": adsl.dtypes.astype(str), "missing": adsl.isna().sum(), "distinct": adsl.nunique()})
summary

## Population and treatment arms

In [ ]:
arms = adsl.groupby("ARM").agg(subjects=("USUBJID", "nunique"), safety=("SAFFL", lambda s: (s == "Y").sum()),
                               efficacy=("EFFFL", lambda s: (s == "Y").sum()), completers=("COMPLFL", lambda s: (s == "Y").sum()))
arms.loc["Total"] = arms.sum()
arms

## Data checks
Simple checks a programmer runs before trusting a dataset. Each line should say OK.

In [ ]:
checks = {
    "One row per subject (USUBJID unique)": adsl["USUBJID"].is_unique,
    "Every subject has an arm": adsl["ARM"].notna().all(),
    "Treatment end is not before treatment start": (adsl["TRTEDT"].dropna() >= adsl.loc[adsl["TRTEDT"].notna(), "TRTSDT"]).all(),
    "Age group agrees with AGE": ((adsl["AGE"] < 65).eq(adsl["AGEGR1"] == "<65")).all()
                                 and ((adsl["AGE"] > 80).eq(adsl["AGEGR1"] == ">80")).all(),
    "Subjects who completed the study have no discontinuation reason": adsl.loc[adsl["EOSSTT"] == "COMPLETED", "DCSREAS"].isna().all(),
}
for name, ok in checks.items():
    print("OK  " if ok else "FAIL", name)

# Something worth a question to the study team:
odd = adsl[(adsl["COMPLFL"] == "Y") & (adsl["EOSSTT"] == "DISCONTINUED")]
print(f"\nNote: {len(odd)} subjects are flagged as completers (COMPLFL=Y) but discontinued the study (EOSSTT).")
print("Check the definition of COMPLFL before using it for a completer analysis.")
odd[["USUBJID", "ARM", "COMPLFL", "EOSSTT", "DCSREAS", "TRTDURD"]]

## Demographics by arm

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=adsl, x="AGE", hue="ARM", multiple="stack", bins=15, ax=axes[0]); axes[0].set_title("Age by arm")
sns.countplot(data=adsl, x="ARM", hue="SEX", ax=axes[1]); axes[1].set_title("Sex by arm")
sns.countplot(data=adsl, x="ARM", hue="AGEGR1", ax=axes[2], hue_order=["<65", "65-80", ">80"]); axes[2].set_title("Age group by arm")
for ax in axes: ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.savefig(plots / "demographics_by_arm.png", dpi=150); plt.show()

## Disposition
Who completed, and why did the others stop?

In [ ]:
reasons = adsl.groupby(["ARM", adsl["DCSREAS"].fillna("COMPLETED")]).size().unstack(fill_value=0)
reasons

In [ ]:
reasons.plot(kind="barh", stacked=True, figsize=(12, 4), colormap="tab20")
plt.title("Disposition by arm"); plt.xlabel("Subjects"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig(plots / "disposition_by_arm.png", dpi=150); plt.show()

## Treatment exposure

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=adsl, x="ARM", y="TRTDURD"); plt.title("Treatment duration (days) by arm")
plt.savefig(plots / "exposure_by_arm.png", dpi=150); plt.show()

## Next
Nothing in this notebook writes back to the volume. The next exercise turns ADSL into a table with production and QC programs.